In [ ]:
import time
import threading
import math
import logging
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist
from robot_wrapper import Robot
from flask import Flask, render_template_string, request, jsonify
import subprocess
import re

# ==========================================
# 1. GLOBAL STATE, LOGGING & ROS2 THREAD
# ==========================================
global_robot_command = "NO HAND DETECTED"
thread_active = True

# --- NEW: Terminal Logger ---
robot_logs = ["[System] Terminal initialized... Waiting for robot."]

def log_to_terminal(msg):
    """Adds a timestamped message to the global logs list."""
    timestamp = time.strftime("%H:%M:%S")
    formatted_msg = f"[{timestamp}] {msg}"
    robot_logs.append(formatted_msg)
    # Keep only the last 20 logs so memory doesn't blow up
    if len(robot_logs) > 20:
        robot_logs.pop(0)

class LidarObserver(Node):
    def __init__(self, robot_instance):
        super().__init__('lidar_observer')
        self.robot = robot_instance
        self.subscription = self.create_subscription(LaserScan, '/scan', self.avoiding_obstacles, 10)
        self.safety_latch_engaged = False

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        if not ranges: return
        total_points = len(ranges)
        cone_size = max(1, int(total_points / 12))
        front_cone = ranges[:cone_size] + ranges[-cone_size:]
        valid_distances = [r for r in front_cone if 0.15 < r <= 0.50 and not math.isinf(r) and not math.isnan(r)]
        
        if valid_distances:
            if not self.safety_latch_engaged:
                closest_obstacle = min(valid_distances)
                warn_msg = f"🚨 LiDAR WARNING: Obstacle at {closest_obstacle:.2f}m! Stopping."
                self.get_logger().warn(warn_msg)
                log_to_terminal(warn_msg) # Send to UI Terminal
                self.safety_latch_engaged = True
        else:
            if self.safety_latch_engaged:
                clear_msg = "✅ PATH CLEAR: Safety latch released."
                self.get_logger().info(clear_msg)
                log_to_terminal(clear_msg) # Send to UI Terminal
                self.safety_latch_engaged = False

def background_robot_executor():
    global global_robot_command, thread_active
    if not rclpy.ok(): rclpy.init()
        
    robot = Robot()
    node = LidarObserver(robot)
    executor = rclpy.executors.SingleThreadedExecutor()
    executor.add_node(node)
    executor.add_node(robot)
    
    last_command = None
    last_command_time = time.time()
    
    log_to_terminal("✅ Robot ROS2 nodes successfully started.")

    try:
        while rclpy.ok() and thread_active:
            executor.spin_once(timeout_sec=0.01)
            current_cmd = global_robot_command
            current_time = time.time()

            # --- NEW AVOIDANCE LOGIC ---
            # If LiDAR detects an obstacle, hijack the command to spin
            if node.safety_latch_engaged:
                current_cmd = "AVOID_OBSTACLE"

            # 10Hz command broadcast to prevent robot safety timeouts
            if current_cmd != last_command or (current_time - last_command_time) > 0.1:
                
                if current_cmd != last_command:
                    # Specific print and terminal log for the obstacle avoidance
                    if current_cmd == "AVOID_OBSTACLE":
                        print("turning to avoid obstacle")
                        log_to_terminal("🔄 turning to avoid obstacle")
                    else:
                        print(f"Executing -> Command: {current_cmd}")
                        if current_cmd not in ["HAND DETECTED", "NO HAND DETECTED"]:
                            log_to_terminal(f"Executing command: {current_cmd}")
                
                # Execute the corresponding robot movement
                if current_cmd in ["NO HAND DETECTED", "STOP", "HAND DETECTED"]:
                    robot.set_cmd_vel(0.0, 0.0, 0.0)
                elif current_cmd == "AVOID_OBSTACLE":
                    robot.set_cmd_vel(0.0, -0.4, 0.5) # Spin right until path is clear
                elif current_cmd == "EMERGENCY STOP":
                    robot.set_cmd_vel(0.0, -0.4, 0.5) 
                elif current_cmd == "GO":
                    robot.set_cmd_vel(0.2, 0.0, 0.5)
                elif current_cmd == "BACKWARD":
                    robot.set_cmd_vel(-0.2, 0.0, 0.5)
                elif current_cmd == "LEFT":
                    robot.set_cmd_vel(0.0, 0.2, 0.5)
                elif current_cmd == "RIGHT":
                    robot.set_cmd_vel(0.0, -0.2, 0.5)
                
                last_command = current_cmd
                last_command_time = current_time
            time.sleep(0.02)
    except Exception as e:
        print(f"Thread encountered error: {e}")
        log_to_terminal(f"❌ ERROR: {e}")
    finally:
        executor.remove_node(node)
        executor.remove_node(robot)
        node.destroy_node()

# ==========================================
# 2. FLASK SERVER & HTML FRONTEND
# ==========================================
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app = Flask(__name__)

html_page = """
<!DOCTYPE html>
<html>
<head>
    <title>Turtlebot Hand Controller</title>
    <style>
        body { 
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
            background-color: #FFF0F5; 
            color: #121212; 
            text-align: center; 
            margin-top: 40px; 
        }
        .container { 
            border: 1px solid #333; 
            padding: 30px; 
            border-radius: 16px; 
            background-color: #8100b4; 
            color: #FFF0F5;
            display: inline-block;
            box-shadow: 0 8px 32px rgba(0,0,0,0.5);
            max-width: 700px;
        }
        h2 { margin-top: 0; color: #FFF0F5; font-weight: 500; letter-spacing: 1px; }
        .video-wrapper { 
            position: relative; 
            width: 640px; 
            height: 480px; 
            margin: 0 auto; 
            background: #FFF0F5; 
            border-radius: 12px;
            overflow: hidden;
            border: 2px solid #f3dcfc;
        }
        video, canvas { position: absolute; left: 0; top: 0; transform: scaleX(-1); }
        #status-container {
            margin-top: 25px;
            padding: 15px;
            background-color: #2a2a2a;
            border-radius: 8px;
            border-left: 5px solid #00ff00;
        }
        #status { font-size: 28px; font-weight: bold; color: #00ff00; margin: 0; }
        
        /* THE TERMINAL UI */
        .terminal {
            background-color: #0c0c0c;
            color: #00ff00;
            font-family: 'Courier New', Courier, monospace;
            text-align: left;
            padding: 15px;
            margin-top: 20px;
            height: 120px;
            overflow-y: auto;
            border-radius: 8px;
            border: 1px solid #444;
            font-size: 14px;
            line-height: 1.4;
            box-shadow: inset 0 0 10px rgba(0,0,0,0.8);
        }
        
        .guide { display: flex; justify-content: center; gap: 20px; margin-top: 20px; font-size: 14px; color: #FFF0F5; }
    </style>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/drawing_utils/drawing_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>
</head>
<body>
    <div class="container">
        <h2>Turtlebot Hand Tracking Feed</h2>
        <div class="video-wrapper">
            <video id="webcam" autoplay playsinline width="640" height="480"></video>
            <canvas id="output_canvas" width="640" height="480"></canvas>
        </div>
        <div id="status-container">
            <div id="status">⏳ System Starting...</div>
        </div>
        
        <div class="terminal" id="terminal-box">
            <div>[System] Terminal loading...</div>
        </div>

        <div class="guide">
            <span>✊ Stop</span> | 
            <span>✌️ Backward</span> | 
            <span>✋ Forward</span> |
            <span>👍 Point L/R</span>
        </div>
    </div>

    <script>
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('output_canvas');
        const ctx = canvas.getContext('2d');
        const statusText = document.getElementById('status');
        const terminalBox = document.getElementById('terminal-box');
        
        let lastSentGesture = "";
        let lastSentTime = 0;

        const iconMap = {
            "STOP": "✊ STOP",
            "BACKWARD": "✌️ BACKWARD",
            "GO": "✋ GO FORWARD",
            "LEFT": "👈 TURN LEFT",
            "RIGHT": "👉 TURN RIGHT",
            "NO HAND DETECTED": "❌ NO HAND DETECTED",
            "HAND DETECTED": "✊ STOP "
        };

        // Fetch logs for the terminal every 1 second
        setInterval(async () => {
            try {
                const response = await fetch('/get_logs');
                const data = await response.json();
                if (data.logs) {
                    terminalBox.innerHTML = data.logs.map(log => `<div>${log}</div>`).join('');
                    terminalBox.scrollTop = terminalBox.scrollHeight;
                }
            } catch (err) {
                console.error("Could not fetch logs");
            }
        }, 1000);

        async function initCamera() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ video: true });
                video.srcObject = stream;
            } catch (err) {
                statusText.innerText = "Error: Cannot access webcam.";
                statusText.style.color = "#ff4444";
                document.getElementById('status-container').style.borderLeftColor = "#ff4444";
            }
        }

        const hands = new Hands({
            locateFile: (file) => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${file}`
        });

        hands.setOptions({
            maxNumHands: 1,
            modelComplexity: 1,
            minDetectionConfidence: 0.5,
            minTrackingConfidence: 0.5
        });

        function evaluateGesture(landmarks) {
            const wrist = landmarks[0];
            function hypot(dx, dy) { return Math.sqrt(dx*dx + dy*dy); }
            function isExtended(tipIdx, knuckleIdx) {
                const distTip = hypot(landmarks[tipIdx].x - wrist.x, landmarks[tipIdx].y - wrist.y);
                const distKnuckle = hypot(landmarks[knuckleIdx].x - wrist.x, landmarks[knuckleIdx].y - wrist.y);
                return distTip > distKnuckle * 1.15;
            }
            
            const indexExt = isExtended(8, 5);
            const middleExt = isExtended(12, 9);
            const ringExt = isExtended(16, 13);
            const pinkyExt = isExtended(20, 17);

            const openFingers = (indexExt?1:0) + (middleExt?1:0) + (ringExt?1:0) + (pinkyExt?1:0);
                                
            const thumbExt = hypot(landmarks[4].x - wrist.x, landmarks[4].y - wrist.y) > 
                             hypot(landmarks[2].x - wrist.x, landmarks[2].y - wrist.y) * 1.1;

            if (openFingers === 0 && !thumbExt) return "STOP";

            if (openFingers === 0 && thumbExt) {
                const thumbDiffX = landmarks[4].x - wrist.x;
                if (thumbDiffX < -0.05) return "RIGHT";
                if (thumbDiffX > 0.05) return "LEFT";
                return "STOP"; 
            }

            if (indexExt && middleExt && !ringExt && !pinkyExt) return "BACKWARD";
            
            if (openFingers >= 3) {
                const fingertipsX = (landmarks[8].x + landmarks[12].x + landmarks[16].x + landmarks[20].x) / 4.0;
                const diffX = fingertipsX - wrist.x;
                if (diffX < -0.05) return "RIGHT";
                if (diffX > 0.05) return "LEFT";
                return "GO";
            }
            
            return "HAND DETECTED";
        }

        hands.onResults((results) => {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.drawImage(results.image, 0, 0, canvas.width, canvas.height);
            let currentGesture = "NO HAND DETECTED";
            
            if (results.multiHandLandmarks && results.multiHandLandmarks.length > 0) {
                const landmarks = results.multiHandLandmarks[0];
                drawConnectors(ctx, landmarks, HAND_CONNECTIONS, {color: '#00FF00', lineWidth: 4});
                drawLandmarks(ctx, landmarks, {color: '#FF0000', lineWidth: 2});
                currentGesture = evaluateGesture(landmarks);
            }
            
            statusText.innerText = iconMap[currentGesture] || currentGesture;
            
            const now = Date.now();
            
            // Re-send command every 300ms to prevent robot stopping due to dropped network packets
            if (currentGesture !== lastSentGesture || (now - lastSentTime) > 300) {
                fetch('/set_command', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ command: currentGesture })
                }).catch(err => console.error("Network hiccup:", err));
                
                lastSentGesture = currentGesture;
                lastSentTime = now;
            }
        });

        const camera = new Camera(video, {
            onFrame: async () => { await hands.send({image: video}); },
            width: 640, height: 480
        });

        initCamera().then(() => camera.start());
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(html_page)

@app.route('/set_command', methods=['POST'])
def set_command():
    global global_robot_command
    data = request.json
    if data and 'command' in data:
        global_robot_command = data['command']
    return jsonify({"status": "success"})

# Flask route to supply logs to the UI terminal
@app.route('/get_logs', methods=['GET'])
def get_logs():
    return jsonify({"logs": robot_logs})

# ==========================================
# 3. MAIN EXECUTION (With No-Key SSH Tunnel)
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🤖 STARTING HYBRID SERVER SYSTEM")
    
    worker_thread = threading.Thread(target=background_robot_executor)
    worker_thread.daemon = True
    worker_thread.start()
    
    print("🌍 Opening global tunnel via localhost.run...")
    
    tunnel_proc = subprocess.Popen(
        ['ssh', '-R', '80:localhost:5000', '-o', 'StrictHostKeyChecking=no', 'nokey@localhost.run'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    public_url = "⚠️ URL Not Found (Check terminal output)"
    
    # Read output to find URL, avoiding the admin link
    for _ in range(30):
        line = tunnel_proc.stdout.readline()
        match = re.search(r'(https://[a-zA-Z0-9.-]+\.(?:lhr\.life|lhr\.run|localhost\.run))', line)
        
        if match:
            found_url = match.group(1)
            if "admin.localhost.run" not in found_url:
                public_url = found_url
                break
                
    # Background thread to constantly empty SSH logs so the pipe doesn't overflow and freeze
    def empty_ssh_bucket():
        try:
            for _ in tunnel_proc.stdout:
                pass
        except:
            pass
            
    bucket_thread = threading.Thread(target=empty_ssh_bucket)
    bucket_thread.daemon = True
    bucket_thread.start()
            
    print(f"🌍 GLOBAL LINK READY: {public_url}")
    print("👉 Open this link on ANY device to control the robot!")
    print("👉 Stop this cell (Square Button) to shut everything down cleanly.")
    print("="*50 + "\n")
    
    try:
        app.run(host='0.0.0.0', port=5000, threaded=True)
    except KeyboardInterrupt:
        pass
    finally:
        print("\n🛑 Shutting down server and closing tunnel...")
        thread_active = False
        
        try:
            tunnel_proc.terminate()
            tunnel_proc.wait(timeout=2)
        except:
            tunnel_proc.kill()
            
        worker_thread.join()
        print("✅ Shutdown complete.")


🤖 STARTING HYBRID SERVER SYSTEM
🌍 Opening global tunnel via localhost.run...
Executing -> Command: NO HAND DETECTED
🌍 GLOBAL LINK READY: https://20e36390b06700.lhr.life
👉 Open this link on ANY device to control the robot!
👉 Stop this cell (Square Button) to shut everything down cleanly.

 * Serving Flask app '__main__'
 * Debug mode: off


[WARN] [1785334217.817731673] [lidar_observer]: 🚨 LiDAR WARNING: Obstacle at 0.47m! Stopping.


turning to avoid obstacle
Executing -> Command: NO HAND DETECTED


[INFO] [1785334221.586408273] [lidar_observer]: ✅ PATH CLEAR: Safety latch released.


Executing -> Command: LEFT
Executing -> Command: RIGHT
Executing -> Command: NO HAND DETECTED
Executing -> Command: LEFT
turning to avoid obstacle


[WARN] [1785334230.910818784] [lidar_observer]: 🚨 LiDAR WARNING: Obstacle at 0.50m! Stopping.
[INFO] [1785334234.166291688] [lidar_observer]: ✅ PATH CLEAR: Safety latch released.


... (13 outputs left out)

[WARN] [1785334489.605960394] [lidar_observer]: 🚨 LiDAR WARNING: Obstacle at 0.35m! Stopping.


Executing -> Command: GO


[INFO] [1785334490.206352680] [lidar_observer]: ✅ PATH CLEAR: Safety latch released.


Executing -> Command: NO HAND DETECTED
Executing -> Command: HAND DETECTED
Executing -> Command: NO HAND DETECTED
Executing -> Command: HAND DETECTED
Executing -> Command: NO HAND DETECTED
Executing -> Command: GO
Executing -> Command: NO HAND DETECTED
Executing -> Command: BACKWARD
Executing -> Command: NO HAND DETECTED


In [ ]:
from pyngrok import ngrok
#ngrok.kill()

In [ ]:
pip install subprocess